# Download all selected-activation batches

Download every activation batch produced by `scripts/run_activation_caching_all_selected_acts.sh`. The notebook is standalone and can run in Colab without installing or cloning `temporal_manifolds`. It maps each generated GCS prefix to a separate local directory. Existing complete files are skipped unless `OVERWRITE` is enabled.

## 1. Setup

In Colab, authenticate with the next cell. In a cloned repository, the notebook also reads the same optional `.env` file as the activation-caching scripts.

In [ ]:
# Run this cell in Colab to authenticate with Google Cloud.
try:
    from google.colab import auth
except ModuleNotFoundError:
    print('Not running in Colab; using existing Application Default Credentials.')
else:
    auth.authenticate_user()

In [ ]:
import os
from collections import Counter
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path, PurePosixPath

from google.cloud import storage
from tqdm.auto import tqdm

repo_root = Path.cwd()
if repo_root.name == 'notebooks':
    repo_root = repo_root.parent
try:
    from dotenv import load_dotenv
except ModuleNotFoundError:
    pass
else:
    load_dotenv(repo_root / '.env')

## 2. Configure the download

By default, every registered dataset is downloaded below `data/all_selected_activations/`. Set `SELECTED_DATASETS` to a tuple of dataset names to download only a subset.

In [ ]:
PROJECT_ID = os.getenv('GCP_PROJECT_ID', 'temporal-interp-exp')
BUCKET_NAME = os.getenv('GCS_BUCKET_NAME', 'temporal-research-bucket')
LOCAL_ROOT = repo_root / 'data' / 'all_selected_activations'
DOWNLOAD_WORKERS = 8
OVERWRITE = False
SELECTED_DATASETS: tuple[str, ...] | None = None

if DOWNLOAD_WORKERS < 1:
    raise ValueError('DOWNLOAD_WORKERS must be at least 1.')

# Mirrors DATASETS in src/temporal_manifolds/dataset/generate.py.
available_datasets = (
    'conversational',
    'conversational_no_time',
    'event_anchored',
    'abstract',
    'plain_english',
    'plain_long',
    'task_only',
    'indirect_horizon',
)
selected_datasets = available_datasets if SELECTED_DATASETS is None else SELECTED_DATASETS
unknown_datasets = sorted(set(selected_datasets) - set(available_datasets))
if unknown_datasets:
    raise ValueError(f'Unknown datasets: {unknown_datasets}. Available: {available_datasets}')
if not selected_datasets:
    raise ValueError('Select at least one dataset.')

def prefix_for_dataset(dataset: str) -> str:
    return 'selected_acts' if dataset == 'conversational' else f'{dataset}_selected_acts'


dataset_prefixes = {dataset: prefix_for_dataset(dataset) for dataset in selected_datasets}
print(f'Bucket: gs://{BUCKET_NAME}')
print(f'Local root: {LOCAL_ROOT}')
for dataset, prefix in dataset_prefixes.items():
    print(f'  {dataset}: gs://{BUCKET_NAME}/{prefix}')

## 3. Discover remote batches

Only files named `activations_batch_*.pt` directly inside each generated prefix are included. The cell fails if any selected dataset has no remote batches, preventing a silently incomplete download.

In [ ]:
client = storage.Client(project=PROJECT_ID)
bucket = client.bucket(BUCKET_NAME)
manifest = []
missing_prefixes = []

for dataset, prefix in dataset_prefixes.items():
    prefix_path = PurePosixPath(prefix)
    blobs = sorted(
        (
            blob
            for blob in bucket.list_blobs(prefix=f'{prefix}/')
            if PurePosixPath(blob.name).parent == prefix_path
            and PurePosixPath(blob.name).name.startswith('activations_batch_')
            and blob.name.endswith('.pt')
        ),
        key=lambda blob: blob.name,
    )
    if not blobs:
        missing_prefixes.append(f'gs://{BUCKET_NAME}/{prefix}')
        continue
    manifest.extend((dataset, prefix, blob) for blob in blobs)
    remote_bytes = sum(int(blob.size or 0) for blob in blobs)
    print(f'{dataset}: {len(blobs):,} batches, {remote_bytes / 2**30:.2f} GiB')

if missing_prefixes:
    formatted = '\n'.join(f'  - {uri}' for uri in missing_prefixes)
    raise FileNotFoundError(f'No activation batches found under:\n{formatted}')

total_bytes = sum(int(blob.size or 0) for _, _, blob in manifest)
print(f'Total: {len(manifest):,} batches, {total_bytes / 2**30:.2f} GiB')

## 4. Download all batches

A local file is reused when its byte size matches GCS. Missing, incomplete, or explicitly overwritten files are downloaded concurrently.

In [ ]:
def download_batch(item):
    dataset, prefix, blob = item
    destination = LOCAL_ROOT / prefix / PurePosixPath(blob.name).name
    destination.parent.mkdir(parents=True, exist_ok=True)
    expected_size = int(blob.size or 0)

    is_complete = (
        destination.exists()
        and not OVERWRITE
        and (expected_size == 0 or destination.stat().st_size == expected_size)
    )
    if is_complete:
        status = 'skipped'
    else:
        blob.download_to_filename(str(destination))
        status = 'downloaded'

    return {
        'dataset': dataset,
        'prefix': prefix,
        'path': destination,
        'expected_size': expected_size,
        'status': status,
    }


LOCAL_ROOT.mkdir(parents=True, exist_ok=True)
with ThreadPoolExecutor(max_workers=DOWNLOAD_WORKERS) as executor:
    download_results = list(
        tqdm(
            executor.map(download_batch, manifest),
            total=len(manifest),
            desc='Downloading activation batches',
        )
    )

status_counts = Counter(result['status'] for result in download_results)
print(dict(status_counts))

## 5. Verify the local copy

The final cell checks that every discovered object exists locally and has the expected size, then prints the destination and batch count for each dataset.

In [ ]:
missing_local_files = [result['path'] for result in download_results if not result['path'].is_file()]
size_mismatches = [
    result['path']
    for result in download_results
    if result['expected_size']
    and result['path'].is_file()
    and result['path'].stat().st_size != result['expected_size']
]
if missing_local_files or size_mismatches:
    raise RuntimeError(
        f'Download verification failed: {len(missing_local_files)} missing, '
        f'{len(size_mismatches)} size mismatches.'
    )

counts_by_dataset = Counter(result['dataset'] for result in download_results)
for dataset in selected_datasets:
    prefix = dataset_prefixes[dataset]
    print(f'{dataset}: {counts_by_dataset[dataset]:,} files in {LOCAL_ROOT / prefix}')
print(f'Verified {len(download_results):,} local activation batches.')